# Bài 3: Đọc & Ghi dữ liệu — Write Modes, Partitioning, Data Skipping

## Mục tiêu
- Nắm rõ 4 write mode: `append`, `overwrite`, `errorifexists`, `ignore`.
- Dùng `replaceWhere` để overwrite có chọn lọc (selective overwrite).
- Hiểu **partition pruning** và **data skipping** giúp Delta đọc nhanh hơn Parquet thuần thế nào.


## 3.1. Write modes

| Mode | Hành vi khi bảng đã tồn tại |
|---|---|
| `append` | Thêm dữ liệu mới, giữ nguyên dữ liệu cũ |
| `overwrite` | Xoá **toàn bộ** dữ liệu cũ, thay bằng dữ liệu mới (nhưng vẫn giữ lịch sử — time travel vẫn thấy version cũ) |
| `errorifexists` (mặc định của `.save()`) | Ném lỗi nếu bảng/path đã tồn tại |
| `ignore` | Không làm gì nếu bảng/path đã tồn tại |

### Selective overwrite với `replaceWhere`

Thay vì overwrite toàn bộ bảng, có thể overwrite **chỉ những partition/điều kiện thoả mãn**, các phần còn lại giữ nguyên:

```python
(df_new.write.format("delta")
   .mode("overwrite")
   .option("replaceWhere", "order_date = '2024-01-02'")
   .saveAsTable("bai03.orders"))
```

Đây là cách phổ biến để "backfill" lại 1 ngày dữ liệu mà không đụng vào các ngày khác — nhanh hơn nhiều so với đọc toàn bộ bảng rồi overwrite lại (và an toàn hơn UPDATE/DELETE thủ công cho khối lượng lớn).

> Từ Delta 2.x+, mặc định **schema của dữ liệu mới phải nằm trong** dữ liệu đang bị `replaceWhere` (dynamic partition overwrite an toàn hơn).

## 3.2. Partition pruning & Data skipping

- **Partition pruning**: nếu bảng `PARTITIONED BY (country)`, dữ liệu vật lý được chia thành các thư mục con `country=VN/`, `country=US/`... Khi query có `WHERE country = 'VN'`, Spark **chỉ đọc thư mục đó**, bỏ qua hoàn toàn các partition khác — không cần mở file để biết.
- **Data skipping**: với các cột **không** phải partition, mỗi commit trong `_delta_log` vẫn lưu **min/max statistics** của 32 cột đầu tiên (mặc định) cho từng file Parquet. Khi query có filter (`WHERE price > 100`), Delta đọc thống kê trong log để loại các file chắc chắn không khớp — mà **không cần mở file Parquet**. Đây là lý do Delta Lake nhanh hơn Parquet thuần dù cùng định dạng file.

Nguyên tắc chọn partition column: dùng cho cột có **cardinality thấp-vừa** (ngày, quốc gia...) và **hay xuất hiện trong WHERE**. Partition theo cột cardinality quá cao (ví dụ `user_id`) sẽ tạo ra hàng triệu thư mục nhỏ — phản tác dụng (xem thêm Bài 7 về small-file problem).


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai03"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai03-doc-ghi")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 3.3. Ví dụ minh hoạ

In [ ]:
from datetime import date

spark.sql("DROP TABLE IF EXISTS bai03.orders")
rows = [
    (1, "A", 10, date(2024, 1, 1)),
    (2, "B", 20, date(2024, 1, 1)),
    (3, "C", 30, date(2024, 1, 2)),
    (4, "D", 40, date(2024, 1, 3)),
]
df = spark.createDataFrame(rows, ["order_id", "product", "qty", "order_date"])
df.write.format("delta").partitionBy("order_date").saveAsTable("bai03.orders")
spark.sql("SELECT * FROM bai03.orders ORDER BY order_id").show()


In [ ]:
# append: them du lieu ngay 2024-01-04, khong dung gi den cac ngay khac
new_rows = [(5, "E", 50, date(2024, 1, 4))]
spark.createDataFrame(new_rows, ["order_id", "product", "qty", "order_date"]) \
    .write.format("delta").mode("append").saveAsTable("bai03.orders")

spark.sql("SELECT * FROM bai03.orders ORDER BY order_id").show()


In [ ]:
# replaceWhere: chi ghi de partition ngay 2024-01-01, cac ngay khac giu nguyen
fixed_rows = [(1, "A-fixed", 99, date(2024, 1, 1))]
(
    spark.createDataFrame(fixed_rows, ["order_id", "product", "qty", "order_date"])
    .write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "order_date = '2024-01-01'")
    .saveAsTable("bai03.orders")
)
spark.sql("SELECT * FROM bai03.orders ORDER BY order_id").show()
# Chi order_id=1,2 (ngay 01-01) bi thay; ngay 02,03,04 khong doi


In [ ]:
# Xem physical plan de thay partition pruning hoat dong: PartitionFilters trong plan
spark.sql("SELECT * FROM bai03.orders WHERE order_date = '2024-01-02'").explain(True)


## 3.4. Thực hành

**Bài 1** — Tạo bảng `bai03.logs (id INT, level STRING, dt DATE, message STRING)` partition theo `dt`, ghi dữ liệu cho 3 ngày khác nhau (mỗi ngày ≥ 2 dòng).

**Bài 2** — Dùng `append` để thêm dữ liệu cho 1 ngày mới. Kiểm tra `SHOW PARTITIONS` trước/sau.

**Bài 3** — Dùng `replaceWhere` để thay toàn bộ dữ liệu của **1 ngày cụ thể** bằng dữ liệu mới, xác nhận các ngày khác không đổi.

**Bài 4** — Thử ghi `mode("errorifexists")` (hoặc `.save()` không mode) vào path của bảng đã tồn tại — quan sát lỗi xảy ra. Sau đó thử `mode("ignore")` — xác nhận dữ liệu không đổi.

**Bài 5** — Chạy `.explain(True)` cho 1 query có filter theo cột partition (`dt`) và 1 query filter theo cột không phải partition (`level`). So sánh phần `PartitionFilters` / `PushedFilters` trong physical plan của 2 query.


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

In [ ]:
# TODO: Bài 4


### Vùng làm bài — Bài 5

In [ ]:
# TODO: Bài 5


---
## Gợi ý / đáp án tham khảo

In [ ]:
# Dap an Bai 1
from datetime import date
spark.sql("DROP TABLE IF EXISTS bai03.logs")
rows = [
    (1, "INFO", date(2024,2,1), "start"), (2, "ERROR", date(2024,2,1), "fail"),
    (3, "INFO", date(2024,2,2), "ok"), (4, "WARN", date(2024,2,2), "slow"),
    (5, "INFO", date(2024,2,3), "done"), (6, "ERROR", date(2024,2,3), "crash"),
]
spark.createDataFrame(rows, ["id","level","dt","message"]).write.format("delta").partitionBy("dt").saveAsTable("bai03.logs")
spark.sql("SHOW PARTITIONS bai03.logs").show()


In [ ]:
# Dap an Bai 2
before = spark.sql("SHOW PARTITIONS bai03.logs").count()
spark.createDataFrame([(7,"INFO",date(2024,2,4),"new day")], ["id","level","dt","message"]) \
    .write.format("delta").mode("append").saveAsTable("bai03.logs")
after = spark.sql("SHOW PARTITIONS bai03.logs").count()
print(before, "->", after)


In [ ]:
# Dap an Bai 3
(
    spark.createDataFrame([(1,"INFO",date(2024,2,1),"start-v2")], ["id","level","dt","message"])
    .write.format("delta").mode("overwrite")
    .option("replaceWhere", "dt = '2024-02-01'")
    .saveAsTable("bai03.logs")
)
spark.sql("SELECT * FROM bai03.logs ORDER BY dt, id").show()


In [ ]:
# Dap an Bai 4
try:
    spark.createDataFrame([(99,"INFO",date(2024,2,1),"x")], ["id","level","dt","message"]) \
        .write.format("delta").mode("errorifexists").saveAsTable("bai03.logs")
except Exception as e:
    print("Loi nhu du kien:", type(e).__name__)

cnt_before = spark.table("bai03.logs").count()
spark.createDataFrame([(99,"INFO",date(2024,2,1),"x")], ["id","level","dt","message"]) \
    .write.format("delta").mode("ignore").saveAsTable("bai03.logs")
cnt_after = spark.table("bai03.logs").count()
print(cnt_before, cnt_after, "-> khong doi vi mode=ignore")


In [ ]:
# Dap an Bai 5
print("== Filter theo cot partition (dt) ==")
spark.sql("SELECT * FROM bai03.logs WHERE dt = '2024-02-02'").explain(True)
print("== Filter theo cot khong phai partition (level) ==")
spark.sql("SELECT * FROM bai03.logs WHERE level = 'ERROR'").explain(True)
# Query theo dt se co PartitionFilters loai bo thu muc khong khop ngay trong scan
# Query theo level chi co PushedFilters/data skipping dua tren min/max stats trong _delta_log
